# Chapter 9 — Answering Questions over Documents

Hands-On LangChain. Code targets **LangChain 1.x**.

We can now load, split, store, and retrieve documents. This chapter takes the final step of
**Retrieval-Augmented Generation (RAG)**: feed the retrieved chunks *and* the user's question to an LLM
to produce a grounded answer. We build the standard "stuff" chain with LCEL, return the source
documents, look at the map-reduce and refine strategies for many documents, and end on a key
limitation that motivates the next part of the book.

Companion notebook for Chapter 9. Requires a running Ollama service and `MODEL`/`EMBEDDING_MODEL` in `.env`.

## 9.1 The RAG workflow

The flow is: a question comes in, we retrieve the relevant chunks, then pass those chunks together with
the question to the LLM, which produces the answer. By default we put all retrieved chunks into one
prompt — the **stuff** strategy — which needs just one LLM call.

First, let's build a small vector store to query (a handful of course-lecture chunks).

In [ ]:
from model_provider import get_chat_model, get_embeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

docs = [
    Document(page_content="I assume familiarity with basic probability and statistics: random "
             "variables, expectation, and variance. If it has been a while, we will review these in "
             "the discussion sections. I also assume basic linear algebra — matrices and vectors.",
             metadata={"source": "lecture-01", "page": 4}),
    Document(page_content="This course is about machine learning. In the discussion sections we will "
             "sometimes refresh prerequisites such as statistics and algebra, and later cover "
             "extensions of the main lecture material.",
             metadata={"source": "lecture-01", "page": 3}),
    Document(page_content="The class is not very programming intensive, but we will do some "
             "programming, mostly in MATLAB or Octave.",
             metadata={"source": "lecture-01", "page": 5}),
    Document(page_content="We will introduce a classification algorithm, which is useful for binary "
             "outcomes such as medical diagnoses or predicting whether a house will sell.",
             metadata={"source": "lecture-02", "page": 2}),
    Document(page_content="I strongly encourage you to form study groups to work through the problem "
             "sets together.",
             metadata={"source": "lecture-01", "page": 2}),
]
embedding = get_embeddings()
vectordb = Chroma.from_documents(docs, embedding)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
llm = get_chat_model(temperature=0)
print("vector store ready:", vectordb._collection.count(), "chunks")

## 9.2 A retrieval-QA chain (the "stuff" strategy)

In 1.x there is no `RetrievalQA` class; we assemble the chain ourselves with LCEL. The key parts are a
**prompt** with `{context}` and `{question}` slots, a function to format the retrieved docs into the
context, and the model. We use a custom prompt that constrains the answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

template = """Use the following pieces of context to answer the question at the end. If you don't
know the answer, just say that you don't know — don't try to make one up. Use three sentences
maximum and keep the answer concise. Always say "thanks for asking!" at the end of the answer.

{context}

Question: {question}
Helpful Answer:"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# answer-from-context sub-chain
answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt | llm | StrOutputParser()
)
# full chain: retrieve, keep the source docs, and attach the answer
qa_chain = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=answer_chain)

In [ ]:
result = qa_chain.invoke("Is probability a class topic?")
print(result["answer"])

Because we kept the retrieved chunks in `context`, we can inspect exactly which sources grounded the answer:

In [ ]:
print("Retrieved from:")
for d in result["context"]:
    print(" -", d.metadata, "|", d.page_content[:60], "...")

## 9.3 Strategies for many documents: map-reduce and refine

"Stuff" puts every chunk in one prompt — simple and cheap, but it breaks when there are too many chunks
to fit the context window. Two classic alternatives scale past that limit.

### Map-reduce

**Map** each document to its own partial answer, then **reduce** the partials into one final answer. It
handles arbitrarily many documents (and parallelizes the map step), but each document is seen in
isolation, so answers split across documents can be missed.

In [ ]:
retrieved = retriever.invoke("Is probability a class topic?")

map_prompt = ChatPromptTemplate.from_template(
    "Using ONLY this context, answer the question (say 'no relevant info' if absent).\n\n"
    "Context: {context}\n\nQuestion: {question}"
)
map_chain = map_prompt | llm | StrOutputParser()

# map: one partial answer per document (run in parallel with .batch)
partials = map_chain.batch([{"context": d.page_content, "question": "Is probability a class topic?"}
                            for d in retrieved])

reduce_prompt = ChatPromptTemplate.from_template(
    "Combine these partial answers into one concise final answer.\n\n"
    "Partial answers:\n{answers}\n\nQuestion: {question}"
)
reduce_chain = reduce_prompt | llm | StrOutputParser()
final = reduce_chain.invoke({"answers": "\n---\n".join(partials), "question": "Is probability a class topic?"})
print(final)

### Refine

**Refine** processes documents sequentially: it drafts an answer from the first document, then updates
that answer with each subsequent document. It carries information across documents better than
map-reduce, at the cost of being sequential (no parallelism).

In [ ]:
refine_prompt = ChatPromptTemplate.from_template(
    "You have an existing answer:\n{answer}\n\nRefine it (only if needed) using this extra "
    "context:\n{context}\n\nQuestion: {question}"
)
refine_chain = refine_prompt | llm | StrOutputParser()

answer = ""
for i, d in enumerate(retrieved):
    if i == 0:
        answer = map_chain.invoke({"context": d.page_content, "question": "Is probability a class topic?"})
    else:
        answer = refine_chain.invoke({"answer": answer, "context": d.page_content,
                                      "question": "Is probability a class topic?"})
print(answer)

## 9.4 A key limitation: no memory

A great feature of chatbots is follow-up questions. But our QA chain has **no memory** — each call is
independent. Watch what happens when a follow-up refers to the previous answer.

In [ ]:
q1 = "Is probability a class topic?"
print("Q1:", q1)
print(qa_chain.invoke(q1)["answer"])

In [ ]:
q2 = "why are those prerequisites needed?"   # "those" refers to Q1 — but the chain can't know that
print("Q2:", q2)
print(qa_chain.invoke(q2)["answer"])

The follow-up's "those prerequisites" has no meaning to the chain, because it never saw the first
exchange — it simply retrieves and answers `q2` on its own. To hold a real conversation over your
documents, you need to combine retrieval with **memory** (Chapter 4) and, more generally, with the
stateful **agents** we build in Part III.

This closes Part II: you can now load, split, embed, retrieve, and answer questions over your own
documents — a complete RAG pipeline.